In [5]:
import cv2
import numpy as np

sift = cv2.SIFT_create()

def count_keypoints(imgs):
    counts = []
    for img in imgs:
        if img.dtype != np.uint8:
            img = img.astype(np.uint8)
        kp, desc = sift.detectAndCompute(img, None)
        counts.append(0 if kp is None else len(kp))
    return np.array(counts)

counts = count_keypoints(train_imgs)  # train_imgs from your Dataset

print("num images:", len(counts))
print("min keypoints:", counts.min())
print("max keypoints:", counts.max())
print("mean keypoints:", counts.mean())
print("median keypoints:", np.median(counts))
print("total keypoints:", counts.sum())


NameError: name 'train_imgs' is not defined

In [4]:
from sklearn.preprocessing import LabelEncoder
from dataset import Dataset
from siftbovw import SiftBoVW
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

PARAM = {
    'n_words': 100,
    'max_sample': 100000,
    'batch_size': 100,
    'random_state': 0
}

N_NEIGHBORS = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
# le = LabelEncoder()
training = Dataset(r"training",img_size=(256, 256))
training.shuffle_data()
imgs, labels = training.get_data()
split_ratio = 0.9
split_index = int(len(imgs) * split_ratio)
imgs_train, labels_train = imgs[:split_index], labels[:split_index]
imgs_test, labels_test = imgs[split_index:], labels[split_index:]


print("Parameters:", PARAM)

sift_bovw = SiftBoVW(n_words=PARAM['n_words'], max_sample=PARAM['max_sample'], batch_size=PARAM['batch_size'], random_state=PARAM['random_state'])
x_train = sift_bovw.fit_transform(imgs_train)

# Check the cluster centers
center = sift_bovw.kmeans.cluster_centers_

x_test = sift_bovw.transform(imgs_test)
for N_NEIGHBOR in N_NEIGHBORS:
    print("Number of neighbors:", N_NEIGHBOR)
    knn = KNeighborsClassifier(n_neighbors= N_NEIGHBOR)
    knn.fit(x_train, labels_train)
    y_pred = knn.predict(x_test)
    accuracy = accuracy_score(labels_test, y_pred)
    print(f"Accuracy: {accuracy*100:.2f}%")


Parameters: {'n_words': 100, 'max_sample': 100000, 'batch_size': 100, 'random_state': 0}
[SIFTBoVW] Total descriptors before sampling: 696266
[SIFTBoVW] Subsampled to 100000 descriptors
Init 1/1 with method k-means++
Inertia for init 1/1: 39485136.0
[MiniBatchKMeans] Reassigning 44 cluster centers.
Minibatch step 1/100000: mean batch inertia: 136766.99
Minibatch step 2/100000: mean batch inertia: 123747.31, ewa inertia: 123747.31
Minibatch step 3/100000: mean batch inertia: 110162.14, ewa inertia: 123720.13993170069
Minibatch step 4/100000: mean batch inertia: 108654.96, ewa inertia: 123690.00987313787
Minibatch step 5/100000: mean batch inertia: 107996.44, ewa inertia: 123658.62304725985
Minibatch step 6/100000: mean batch inertia: 104236.99, ewa inertia: 123619.78016959412
Minibatch step 7/100000: mean batch inertia: 100391.94, ewa inertia: 123573.3249538071
Minibatch step 8/100000: mean batch inertia: 100902.67, ewa inertia: 123527.98409730806
Minibatch step 9/100000: mean batch ine

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.kernel_approximation import AdditiveChi2Sampler
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

print("=" * 60)
print("Non-linear SVM with Homogeneous Kernel Map")
print("=" * 60)

# Homogeneous Kernel Map + Linear SVM (optimized version)
print("\nAdditiveChi2Sampler + StandardScaler + LinearSVC:")
SAMPLE_STEPS = [2, 3]  # Reduce parameter combinations, focus on effective ones
C_VALUES = [0.01, 0.1, 1, 10, 100]  # Expand C value range

for sample_steps in SAMPLE_STEPS:
    print(f"\n  Sample steps: {sample_steps}")
    for C in C_VALUES:
        # Build optimized pipeline: Kernel Map -> Standardization -> Linear SVM
        pipeline = Pipeline([
            ('chi2_sampler', AdditiveChi2Sampler(sample_steps=sample_steps)),
            ('scaler', StandardScaler(with_mean=False)),  # No centering for sparse data
            ('svm', LinearSVC(C=C, max_iter=5000, dual='auto', random_state=PARAM['random_state']))
        ])
        pipeline.fit(x_train, labels_train)
        y_pred = pipeline.predict(x_test)
        accuracy = accuracy_score(labels_test, y_pred)
        print(f"    C={C}: Accuracy: {accuracy*100:.2f}%")


Non-linear SVM with Homogeneous Kernel Map

AdditiveChi2Sampler + StandardScaler + LinearSVC:

  Sample steps: 2
    C=0.01: Accuracy: 41.33%
    C=0.1: Accuracy: 40.00%
    C=1: Accuracy: 38.00%
    C=10: Accuracy: 36.67%
    C=100: Accuracy: 36.00%

  Sample steps: 3
    C=0.01: Accuracy: 42.67%
    C=0.1: Accuracy: 39.33%
    C=1: Accuracy: 37.33%
    C=10: Accuracy: 34.67%
    C=100: Accuracy: 34.67%


In [ ]:
from phow import PHOW
from sklearn.svm import LinearSVC
from sklearn.kernel_approximation import AdditiveChi2Sampler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

print("=" * 60)
print("PHOW (Pyramid Histogram of Words) Features")
print("=" * 60)

# Configure parameters
PHOW_PARAM = {
    'n_words': 250,
    'max_sample': 200000,
    'batch_size': 100,
    'pyramid_levels': [0, 1, 2],  # 1x1, 2x2, 4x4 grids
    'random_state': 0
}

print(f"\nParameters: {PHOW_PARAM}")

# Train PHOW feature extractor
phow = PHOW(
    n_words=PHOW_PARAM['n_words'],
    max_sample=PHOW_PARAM['max_sample'],
    batch_size=PHOW_PARAM['batch_size'],
    pyramid_levels=PHOW_PARAM['pyramid_levels'],
    random_state=PHOW_PARAM['random_state']
)

# Extract features
print("\nExtracting training set features...")
x_train_phow = phow.fit_transform(imgs_train)
print(f"Training set feature shape: {x_train_phow.shape}")

print("\nExtracting test set features...")
x_test_phow = phow.transform(imgs_test)
print(f"Test set feature shape: {x_test_phow.shape}")

# Use Homogeneous Kernel Map + LinearSVC
print("\n" + "=" * 60)
print("PHOW + Chi2 Kernel Map + LinearSVC")
print("=" * 60)

SAMPLE_STEPS = [2]
C_VALUES = [0.01, 0.1, 1, 10]

best_acc = 0
best_params = {}

for sample_steps in SAMPLE_STEPS:
    print(f"\n  Sample steps: {sample_steps}")
    for C in C_VALUES:
        pipeline = Pipeline([
            ('chi2_sampler', AdditiveChi2Sampler(sample_steps=sample_steps)),
            ('svm', LinearSVC(C=C, max_iter=3000, dual='auto', random_state=PHOW_PARAM['random_state']))
        ])
        pipeline.fit(x_train_phow, labels_train)
        y_pred = pipeline.predict(x_test_phow)
        accuracy = accuracy_score(labels_test, y_pred)
        print(f"    C={C}: Accuracy: {accuracy*100:.2f}%")
        
        if accuracy > best_acc:
            best_acc = accuracy
            best_params = {'sample_steps': sample_steps, 'C': C}

print(f"\nBest accuracy: {best_acc*100:.2f}%")
print(f"Best parameters: {best_params}")


PHOW (Pyramid Histogram of Words) Features

Parameters: {'n_words': 250, 'max_sample': 200000, 'batch_size': 100, 'pyramid_levels': [0, 1, 2], 'random_state': 0}

Extracting training set features...
[PHOW] 提取 Dense SIFT 描述符...
[PHOW] 总描述符数: 1382400
[PHOW] 采样到 200000 个描述符
[PHOW] 训练 K-Means (n_clusters=250)...
Init 1/1 with method k-means++
Inertia for init 1/1: 36999828.0
[MiniBatchKMeans] Reassigning 50 cluster centers.
Minibatch step 1/200000: mean batch inertia: 120321.41
[MiniBatchKMeans] Reassigning 50 cluster centers.
Minibatch step 2/200000: mean batch inertia: 122714.93, ewa inertia: 122714.93
[MiniBatchKMeans] Reassigning 50 cluster centers.
Minibatch step 3/200000: mean batch inertia: 110993.27, ewa inertia: 122703.208398608
[MiniBatchKMeans] Reassigning 50 cluster centers.
Minibatch step 4/200000: mean batch inertia: 108012.89, ewa inertia: 122688.51815366061
[MiniBatchKMeans] Reassigning 50 cluster centers.
Minibatch step 5/200000: mean batch inertia: 115027.61, ewa inertia:

In [2]:
from gist_feature import GISTFeature
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np

print("=" * 60)
print("GIST Global Scene Features")
print("=" * 60)

# Configure GIST parameters
gist = GISTFeature(n_scales=4, n_orientations=8, n_blocks=4)
print(f"GIST feature dimension: {gist.feature_dim}")

# Extract GIST features
print("\nExtracting training set GIST features...")
x_train_gist = gist.extract(imgs_train)
print(f"Training set GIST feature shape: {x_train_gist.shape}")

print("\nExtracting test set GIST features...")
x_test_gist = gist.extract(imgs_test)
print(f"Test set GIST feature shape: {x_test_gist.shape}")

# Test different classifiers
print("\n" + "=" * 60)
print("GIST + Different Classifiers")
print("=" * 60)

# 1. Linear SVM
print("\n[1] GIST + LinearSVC:")
C_VALUES = [0.01, 0.1, 1, 10]
for C in C_VALUES:
    svm = LinearSVC(C=C, max_iter=2000, random_state=PARAM['random_state'])
    svm.fit(x_train_gist, labels_train)
    y_pred = svm.predict(x_test_gist)
    accuracy = accuracy_score(labels_test, y_pred)
    print(f"  C={C}: Accuracy: {accuracy*100:.2f}%")

# 2. Random Forest
print("\n[2] GIST + Random Forest:")
N_ESTIMATORS = [50, 100, 200]
for n_est in N_ESTIMATORS:
    rf = RandomForestClassifier(n_estimators=n_est, random_state=PARAM['random_state'], n_jobs=-1)
    rf.fit(x_train_gist, labels_train)
    y_pred = rf.predict(x_test_gist)
    accuracy = accuracy_score(labels_test, y_pred)
    print(f"  n_estimators={n_est}: Accuracy: {accuracy*100:.2f}%")


GIST Global Scene Features
GIST feature dimension: 512

Extracting training set GIST features...


NameError: name 'imgs_train' is not defined

In [3]:
from sklearn.preprocessing import normalize
from sklearn.svm import LinearSVC
from sklearn.kernel_approximation import AdditiveChi2Sampler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import numpy as np

print("=" * 60)
print("Feature Fusion: PHOW + GIST")
print("=" * 60)

# Concatenate PHOW and GIST features
print("\nConcatenating features...")
x_train_combined = np.hstack([x_train_phow, x_train_gist])
x_test_combined = np.hstack([x_test_phow, x_test_gist])

print(f"Combined training set feature shape: {x_train_combined.shape}")
print(f"Combined test set feature shape: {x_test_combined.shape}")

# L2 normalization
x_train_combined = normalize(x_train_combined, norm='l2')
x_test_combined = normalize(x_test_combined, norm='l2')

# Use LinearSVC
print("\n" + "=" * 60)
print("Combined Features + LinearSVC")
print("=" * 60)

C_VALUES = [0.01, 0.1, 1, 10]
best_acc = 0
best_C = 0

for C in C_VALUES:
    svm = LinearSVC(C=C, max_iter=3000, dual='auto', random_state=PARAM['random_state'])
    svm.fit(x_train_combined, labels_train)
    y_pred = svm.predict(x_test_combined)
    accuracy = accuracy_score(labels_test, y_pred)
    print(f"  C={C}: Accuracy: {accuracy*100:.2f}%")
    
    if accuracy > best_acc:
        best_acc = accuracy
        best_C = C

print(f"\nBest accuracy: {best_acc*100:.2f}%")
print(f"Best C value: {best_C}")

# Use Chi2 Kernel Map
print("\n" + "=" * 60)
print("Combined Features + Chi2 Kernel Map + LinearSVC")
print("=" * 60)

SAMPLE_STEPS = [2]
C_VALUES = [0.01, 0.1, 1]

for sample_steps in SAMPLE_STEPS:
    print(f"\n  Sample steps: {sample_steps}")
    for C in C_VALUES:
        pipeline = Pipeline([
            ('chi2_sampler', AdditiveChi2Sampler(sample_steps=sample_steps)),
            ('svm', LinearSVC(C=C, max_iter=3000, dual='auto', random_state=PARAM['random_state']))
        ])
        pipeline.fit(x_train_combined, labels_train)
        y_pred = pipeline.predict(x_test_combined)
        accuracy = accuracy_score(labels_test, y_pred)
        print(f"    C={C}: Accuracy: {accuracy*100:.2f}%")


Feature Fusion: PHOW + GIST

Concatenating features...


NameError: name 'x_train_phow' is not defined